# Frozen NeuroLM/LaBraM EEG-only sentiment probe (ZuCo Task 1)

**Question:** do pretrained NeuroLM/LaBraM EEG representations carry enough sentiment information to classify ZuCo sentences above chance?

This notebook only runs code from the repository (`src/neurolm`, `scripts/`). It does **not** train a fusion model. Run it top to bottom:

1. clone the repository and NeuroLM, install dependencies
2. mount Drive and set the data locations
3. download the pinned NeuroLM-B checkpoint (once, to Drive)
4. run the test suite
5. inspect raw ZuCo EEG
6. channel-mapping diagnostics
7. tiny smoke test (1 subject, 10 sentences)
8. extract frozen embeddings for all trials (cached, resumable)
9. frozen linear probes under the three split protocols
10. evaluate
11. sanity checks, including shuffled labels
12. build the report and save everything to Drive

A GPU runtime speeds up step 8; every other step runs on CPU.

## 1. Repository, NeuroLM, and dependencies

In [ ]:
import os, subprocess

REPO_URL = 'https://github.com/MohammadJRanjbar/zuco-multimodal-sentiment.git'
BRANCH = 'feature/neurolm-eeg-probe'
REPO_DIR = '/content/zuco-multimodal-sentiment'
NEUROLM_URL = 'https://github.com/935963004/NeuroLM.git'
NEUROLM_COMMIT = '0cda987'  # code version the probe was written against
NEUROLM_DIR = '/content/NeuroLM'

if not os.path.exists(REPO_DIR):
    !git clone --branch $BRANCH $REPO_URL $REPO_DIR
else:
    !git -C $REPO_DIR fetch origin && git -C $REPO_DIR checkout $BRANCH && git -C $REPO_DIR pull --ff-only
if not os.path.exists(NEUROLM_DIR):
    !git clone $NEUROLM_URL $NEUROLM_DIR
!git -C $NEUROLM_DIR checkout -q $NEUROLM_COMMIT
%cd $REPO_DIR
!pip install -q -r requirements-neurolm.txt
!git log --oneline -1

## 2. Drive and data locations

Set `USE_DRIVE = False` to use files uploaded to `/content/data` instead.

In [ ]:
USE_DRIVE = True
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    THESIS_ROOT = '/content/drive/MyDrive/Thesis'
else:
    THESIS_ROOT = '/content/data/Thesis'

MAT_DIR = f'{THESIS_ROOT}/Data/zuco_og_raw'                      # results*_SR.mat (rawData)
LABELS_CSV = f'{THESIS_ROOT}/Data/zuco_sentiment_labels_task1_fixed.csv'
ARTIFACTS = f'{THESIS_ROOT}/CachedArtifacts/zuco_multimodal_sentiment'
HANDCRAFTED_DIR = f'{ARTIFACTS}/eeg_features/classical_v2_normalized_line_length'
CHECKPOINT_DIR = f'{ARTIFACTS}/models/NeuroLM'
CACHE_DIR = f'{ARTIFACTS}/neurolm_embeddings'                     # fingerprinted embedding cache
RESULTS_BASE = f'{THESIS_ROOT}/Results/zuco_multimodal_sentiment/neurolm_probe'
RUN_TAG = 'v4_frozen_probe'
RUN_DIR = f'{RESULTS_BASE}/{RUN_TAG}'
MAPPING_DIR = f'{RUN_DIR}/mapping'
INSPECT_DIR = f'{RUN_DIR}/inspection'
CONFIG = 'configs/neurolm_probe.yaml'

for path in [MAT_DIR, LABELS_CSV, HANDCRAFTED_DIR]:
    assert os.path.exists(path), f'missing: {path}'
for path in [CHECKPOINT_DIR, CACHE_DIR, RUN_DIR]:
    os.makedirs(path, exist_ok=True)
print(sorted(os.listdir(MAT_DIR)))

# Optional: copy the large .mat files to local disk once per runtime for faster reads.
LOCAL_COPY = True
if LOCAL_COPY:
    LOCAL_MAT = '/content/zuco_og_raw'
    !mkdir -p $LOCAL_MAT && rsync -a --include='results*_SR.mat' --exclude='*' "$MAT_DIR/" "$LOCAL_MAT/"
    MAT_DIR = LOCAL_MAT

## 3. NeuroLM-B checkpoint (pinned revision, downloaded once to Drive)

In [ ]:
import yaml
from huggingface_hub import hf_hub_download

ckpt_cfg = yaml.safe_load(open(CONFIG))['checkpoint']
CHECKPOINT = os.path.join(CHECKPOINT_DIR, ckpt_cfg['filename'])
if not os.path.exists(CHECKPOINT):
    CHECKPOINT = hf_hub_download(ckpt_cfg['repo'], ckpt_cfg['filename'], revision=ckpt_cfg['revision'],
                                 local_dir=CHECKPOINT_DIR)
print(CHECKPOINT, round(os.path.getsize(CHECKPOINT) / 1e9, 2), 'GB')
LOCAL_CHECKPOINT = '/content/NeuroLM-B.pt'
if not os.path.exists(LOCAL_CHECKPOINT):
    !cp "$CHECKPOINT" $LOCAL_CHECKPOINT
CHECKPOINT = LOCAL_CHECKPOINT

## 4. Tests

Includes a forward pass through the real NeuroLM architecture (random weights) that checks batch invariance.

In [ ]:
!python -m pytest -q

## 5. Raw ZuCo inspection

Shapes, subjects, valid/missing trials, durations, NaNs, units, reference state, and the 50 Hz line signature.

In [ ]:
!python scripts/inspect_raw_zuco.py --config $CONFIG --mat-dir "$MAT_DIR" --labels-csv "$LABELS_CSV" \
    --out-dir "$INSPECT_DIR" --report reports/raw_zuco_inspection.md
from IPython.display import Markdown, display
display(Markdown(open('reports/raw_zuco_inspection.md').read()))

## 6. Channel-mapping diagnostics

Adds the checkpoint embedding check, the Cz reference decision from step 5, the token budget, and a montage-consistency test on real trials.

In [ ]:
!python scripts/check_channel_mapping.py --config $CONFIG --out-dir "$MAPPING_DIR" \
    --neurolm-dir $NEUROLM_DIR --checkpoint $CHECKPOINT \
    --inspection-summary "$INSPECT_DIR/raw_zuco_summary.json" \
    --mat-dir "$MAT_DIR" --labels-csv "$LABELS_CSV" --report reports/channel_mapping.md
display(Markdown(open('reports/channel_mapping.md').read()))

## 7. Tiny smoke test: one subject, ten sentences

Verifies mapping → preprocessing → one forward pass → pooled embeddings, into a separate `smoke/` cache.

In [ ]:
!python scripts/extract_neurolm_features.py --config $CONFIG --mat-dir "$MAT_DIR" --labels-csv "$LABELS_CSV" \
    --neurolm-dir $NEUROLM_DIR --checkpoint $CHECKPOINT --cache-dir "$CACHE_DIR" --mapping-dir "$MAPPING_DIR" --smoke

import glob, numpy as np, pandas as pd
for view in sorted(glob.glob(f'{CACHE_DIR}/smoke/*__*')):
    meta = pd.read_csv(f'{view}/metadata.csv')
    emb = np.load(f'{view}/embeddings.npz')
    shapes = {k: emb[k].shape for k in emb.files if k != 'sample_id'}
    finite = all(np.isfinite(emb[k]).all() for k in emb.files if k != 'sample_id')
    print(os.path.basename(view), len(meta), 'trials | finite:', finite, '|', shapes['tokenizer__mean'], shapes['gpt__mean'])
    assert finite and len(meta) > 0

## 8. Extract frozen embeddings for every subject × sentence trial

Resumable: finished subjects are skipped. The cache folder name carries a fingerprint of checkpoint, channels, preprocessing, length handling, and pooling.

In [ ]:
!python scripts/extract_neurolm_features.py --config $CONFIG --mat-dir "$MAT_DIR" --labels-csv "$LABELS_CSV" \
    --neurolm-dir $NEUROLM_DIR --checkpoint $CHECKPOINT --cache-dir "$CACHE_DIR" --mapping-dir "$MAPPING_DIR"

## 9. Frozen linear probes

First a one-seed pipeline check (joint protocol, logistic regression), then the full predeclared matrix: three protocols × {handcrafted: logreg, SVM; NeuroLM: logreg, SVM, MLP, linear layer} × 5 seeds, plus secondary features and views.

In [ ]:
COMMON = f'--config {CONFIG} --cache-dir "{CACHE_DIR}" --handcrafted-dir "{HANDCRAFTED_DIR}" --results-dir "{RESULTS_BASE}"'
!python scripts/run_neurolm_probe.py $COMMON --run-tag {RUN_TAG}_smoke --smoke
!python scripts/run_neurolm_probe.py $COMMON --run-tag $RUN_TAG

## 10. Evaluate

In [ ]:
summary = pd.read_csv(f'{RUN_DIR}/tables/probe_summary.csv')
cols = ['protocol', 'feature', 'view', 'classifier', 'macro_f1_mean', 'macro_f1_std', 'accuracy_mean',
        'balanced_accuracy_mean', 'macro_f1_ci95_sent', 'macro_f1_ci95_subj']
display(summary[cols].round(4))
display(pd.read_csv(f'{RUN_DIR}/tables/paired_neurolm_vs_handcrafted.csv').round(4))

## 11. Sanity checks

Shuffled labels (sentence- and trial-level permutation nulls), shuffled sentence association, subject-ID decoding, Gaussian random features, label-distribution baselines, collapse diagnostics, and a duration-only baseline. `N_PERMUTATIONS` permutations per protocol re-run the whole pipeline; 100 takes roughly 1–2 hours on CPU.

In [ ]:
N_PERMUTATIONS = 100
!python scripts/run_neurolm_sanity.py $COMMON --run-tag $RUN_TAG --n-permutations $N_PERMUTATIONS

## 12. Report and saved results

Everything is already on Drive under `RUN_DIR`. This builds `reports/neurolm_probe_results.md` (also copied to `RUN_DIR`) and copies the regenerated reports next to the results.

In [ ]:
!python scripts/build_neurolm_report.py --config $CONFIG --run-dir "$RUN_DIR" \
    --inspection-summary "$INSPECT_DIR/raw_zuco_summary.json" --mapping-checks "$MAPPING_DIR/mapping_checks.json"
!mkdir -p "$RUN_DIR/reports" && cp reports/*.md "$RUN_DIR/reports/"
display(Markdown(open('reports/neurolm_probe_results.md').read()))
print('saved under', RUN_DIR)